# Notebook 2: Agent Decision Loop (LLM-Powered)
SupplyAgent reads `world_state` filtered by `event_timestamp` at each simulated
decision time, passes the context to Llama 3.3 70B via Databricks Foundation Model APIs,
and writes the LLM-generated decision and reasoning to `agent_memory` with dual
timestamps implementing the bi-temporal model.


In [0]:
import mlflow.deployments
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)
from datetime import datetime, timezone

llm_client = mlflow.deployments.get_deploy_client("databricks")


In [0]:
# Recreate agent_memory clean on every run
# action_timestamp : when the agent decided (event time)
# system_timestamp : when the record was committed to Delta (transaction time)
# These two diverge during data incidents - that is the bi-temporal insight

spark.sql("DROP TABLE IF EXISTS workspace.supply_agent_demo.agent_memory")

spark.sql("""
    CREATE TABLE workspace.supply_agent_demo.agent_memory (
        agent_id           STRING,
        store              STRING,
        decision           STRING,
        reasoning          STRING,
        context_inventory  INT,
        context_temp_f     INT,
        action_timestamp   TIMESTAMP,
        system_timestamp   TIMESTAMP
    )
    USING DELTA
""")


In [0]:
agent_memory_schema = StructType([
    StructField("agent_id",           StringType()),
    StructField("store",              StringType()),
    StructField("decision",           StringType()),
    StructField("reasoning",          StringType()),
    StructField("context_inventory",  IntegerType()),
    StructField("context_temp_f",     IntegerType()),
    StructField("action_timestamp",   TimestampType()),
    StructField("system_timestamp",   TimestampType()),
])


In [0]:
def query_supply_agent(
    decision_time_str: str,
    current_inventory: int,
    current_temp_f: int,
    current_condition: str
) -> tuple:
    """
    Passes the current store context to Llama 3.3 70B via Databricks
    Foundation Model APIs. Returns (decision, reasoning) extracted
    from the structured LLM response.
    """
    prompt = f"""You are SupplyAgent, an autonomous supply chain decision engine
for a Seattle coffee shop chain. Your job is to monitor store inventory and
weather conditions and issue purchase orders when necessary.

Current store context at {decision_time_str}:
- Store: Downtown Seattle
- Oat milk inventory: {current_inventory} gallons
- Weather: {current_temp_f}F, {current_condition}

Supply rules:
- Normal reorder threshold: inventory below 50 gallons triggers a 200 gallon order.
- Emergency threshold: inventory below 0 OR temperature above 90F triggers an 8000 gallon order.
- Otherwise: no action required.

Based strictly on the data provided, state your decision and reasoning.
Respond in exactly this format with no additional text:
DECISION: <your decision>
REASONING: <your reasoning in one sentence>"""

    response = llm_client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 200,
            "temperature": 0
        }
    )

    llm_output = response["choices"][0]["message"]["content"]

    decision_line = next(
        (line for line in llm_output.splitlines() if line.startswith("DECISION:")),
        "DECISION: UNKNOWN"
    )
    reasoning_line = next(
        (line for line in llm_output.splitlines() if line.startswith("REASONING:")),
        "REASONING: Unable to parse LLM response."
    )

    order_decision     = decision_line.replace("DECISION:", "").strip()
    decision_reasoning = reasoning_line.replace("REASONING:", "").strip()

    return order_decision, decision_reasoning


In [0]:
def execute_agent_cycle(decision_time_str: str):
    """
    Simulates one agent cycle at a given decision time.
    Reads the latest world_state row whose event_timestamp
    is at or before the simulated decision time.
    Passes context to the LLM and writes the decision to agent_memory.
    """
    world_state_at_decision = spark.sql(f"""
        SELECT *
        FROM workspace.supply_agent_demo.world_state
        WHERE store = 'Downtown'
          AND event_timestamp <= '{decision_time_str}'
        ORDER BY event_timestamp DESC
        LIMIT 1
    """).collect()[0]

    current_inventory = world_state_at_decision["inventory_gallons"]
    current_temp_f    = world_state_at_decision["weather_temp_f"]
    current_condition = world_state_at_decision["weather_condition"]

    order_decision, decision_reasoning = query_supply_agent(
        decision_time_str,
        current_inventory,
        current_temp_f,
        current_condition
    )

    spark.createDataFrame(
        [(
            "SupplyAgent-v1",
            "Downtown",
            order_decision,
            decision_reasoning,
            int(current_inventory),
            int(current_temp_f),
            datetime.fromisoformat(decision_time_str),
            datetime.now(timezone.utc).replace(tzinfo=None)
        )],
        agent_memory_schema
    ).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.supply_agent_demo.agent_memory")

    print(f"[{decision_time_str}]")
    print(f"  Context   : {current_inventory} gal, {current_temp_f}F, {current_condition}")
    print(f"  Decision  : {order_decision}")
    print(f"  Reasoning : {decision_reasoning}\n")


In [0]:
# Three agent cycles covering the full incident window:
#   07:30 - normal world, no action expected
#   08:14 - during the data corruption window, LLM sees -999 and 102F
#   08:20 - post self-correction, back to normal

execute_agent_cycle("2026-05-19 07:30:00")
execute_agent_cycle("2026-05-19 08:14:00")
execute_agent_cycle("2026-05-19 08:20:00")


In [0]:
print("Full agent memory log:")
spark.sql("""
    SELECT agent_id, store, action_timestamp, decision,
           context_inventory, context_temp_f, reasoning
    FROM workspace.supply_agent_demo.agent_memory
    ORDER BY action_timestamp
""").show(truncate=False)


## Agent Decision Timeline
The chart below plots what the agent saw at each decision point against
what the data actually was. The 08:14 decision stands out immediately.
Without Time Travel, there is no way to explain it from the live table alone.


In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

df_decisions = spark.sql("""
    SELECT action_timestamp, decision, context_inventory, context_temp_f
    FROM workspace.supply_agent_demo.agent_memory
    WHERE store = 'Downtown'
    ORDER BY action_timestamp
""").toPandas()

df_decisions["time_label"] = pd.to_datetime(
    df_decisions["action_timestamp"]
).dt.strftime("%H:%M")

is_anomalous = df_decisions["context_inventory"] < 0

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    "SupplyAgent Decision Log - Three Cycles\n"
    "The 08:14 decision is logical given what the agent saw. "
    "The data it saw was wrong.",
    fontsize=13, fontweight="bold", y=0.98
)

# --- Inventory the agent saw ---
bar_colors = ["#d62728" if anom else "#1f77b4" for anom in is_anomalous]
ax1.bar(df_decisions["time_label"], df_decisions["context_inventory"],
        color=bar_colors, width=0.4, edgecolor="white")
ax1.axhline(y=0, color="#d62728", linestyle="--", linewidth=1, alpha=0.7)
ax1.set_ylabel("Inventory seen (gal)", fontsize=10)
ax1.set_title("What the agent saw: Inventory", fontsize=10)
for i, (val, anom) in enumerate(zip(df_decisions["context_inventory"], is_anomalous)):
    ax1.text(i, val + 3 if val >= 0 else val - 55,
             str(val), ha="center", fontsize=10,
             color="#d62728" if anom else "#1f77b4", fontweight="bold")

# --- Temperature the agent saw ---
temp_colors = ["#d62728" if anom else "#2ca02c" for anom in is_anomalous]
ax2.bar(df_decisions["time_label"], df_decisions["context_temp_f"],
        color=temp_colors, width=0.4, edgecolor="white")
ax2.axhline(y=90, color="#d62728", linestyle="--", linewidth=1, alpha=0.7,
            label="Emergency threshold (90F)")
ax2.set_ylabel("Temperature seen (F)", fontsize=10)
ax2.set_title("What the agent saw: Temperature", fontsize=10)
ax2.legend(fontsize=9)
for i, (val, anom) in enumerate(zip(df_decisions["context_temp_f"], is_anomalous)):
    ax2.text(i, val + 0.5, f"{val}F", ha="center", fontsize=10,
             color="#d62728" if anom else "#2ca02c", fontweight="bold")

# --- Decision outcome ---
decision_labels = df_decisions["decision"].apply(
    lambda d: "EMERGENCY ORDER" if "8000" in d else
              "STANDARD ORDER"  if "200"  in d else
              "NO ACTION"
)
outcome_colors = {
    "EMERGENCY ORDER": "#d62728",
    "STANDARD ORDER":  "#ff7f0e",
    "NO ACTION":       "#2ca02c"
}
bar_outcome_colors = [outcome_colors[label] for label in decision_labels]
ax3.bar(df_decisions["time_label"], [1, 1, 1],
        color=bar_outcome_colors, width=0.4, edgecolor="white")
ax3.set_yticks([])
ax3.set_ylabel("Decision", fontsize=10)
ax3.set_xlabel("Agent cycle time", fontsize=10)
ax3.set_title("Agent Decision Outcome", fontsize=10)
for i, label in enumerate(decision_labels):
    ax3.text(i, 0.5, label, ha="center", va="center",
             fontsize=10, fontweight="bold", color="white")

legend_patches = [
    mpatches.Patch(color="#d62728", label="Emergency order / corrupted reading"),
    mpatches.Patch(color="#2ca02c", label="No action / normal reading"),
]
ax3.legend(handles=legend_patches, fontsize=9, loc="upper right")

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/tmp/agent_decisions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /tmp/agent_decisions.png")
